### Wikicommons depict SPA bilder 
* [#53](https://github.com/salgo60/spa2Commons/issues/53)
* Denna notebook [53_SPA_depict.ipynb](53_SPA_depict.ipynb)

In [1]:
from datetime import datetime
start_time  = datetime.now()
print("Last run: ", start_time)

Last run:  2026-03-07 21:18:24.175751


In [2]:
import requests
import re
import time
from tqdm import tqdm

version = "v2"
COMMONS_API = "https://commons.wikimedia.org/w/api.php"
WD_API = "https://www.wikidata.org/w/api.php"
SPARQL = "https://query.wikidata.org/sparql"

HEADERS = {
    "User-Agent": "SPA-sync-bot/1.0 (salgo60@msn.com)"
}

import json
import os

STATE_FILE = "spa_bot_state.json"
QS_FILE = f"quickstatements_depicts_{version}.txt"

session = requests.Session()
session.headers.update(HEADERS)

CATEGORY = "Category:Swedish_Portrait_Archive"

SPA_REGEX = re.compile(r"portrattarkiv\.se/details/([A-Za-z0-9]+)", re.I)

mediainfo_cache = {} 
skipped_spa = 0
skipped_depicts = 0

def save_state(i, qs, skipped_spa, skipped_depicts):

    state = {
        "index": i,
        "qs_len": len(qs),
        "skipped_spa": skipped_spa,
        "skipped_depicts": skipped_depicts
    }

    with open(STATE_FILE, "w") as f:
        json.dump(state, f) 
start_index = 0
qs = []

if os.path.exists(STATE_FILE):

    with open(STATE_FILE) as f:
        state = json.load(f)

    skipped_spa = state.get("skipped_spa", 0)
    skipped_depicts = state.get("skipped_depicts", 0)

    print("Resuming previous run")

    if os.path.exists(QS_FILE):
        with open(QS_FILE) as f:
            qs = f.read().splitlines()

def get_mediainfo_batch(mids):

    missing = [m for m in mids if m not in mediainfo_cache]

    if not missing:
        return

    params = {
        "action": "wbgetentities",
        "format": "json",
        "ids": "|".join(missing),
        "props": "claims"
    }

    r = safe_get(COMMONS_API, params)
    data = r.json()

    entities = data.get("entities", {})

    for mid in missing:

        entity = entities.get(mid, {})
        claims = entity.get("claims", {})

        mediainfo_cache[mid] = claims 
        
def has_string_claim(claims, prop, value):

    if prop not in claims:
        return False

    for c in claims[prop]:
        v = c["mainsnak"]["datavalue"]["value"]
        if v == value:
            return True

    return False


def has_item_claim(claims, prop, qid):

    if prop not in claims:
        return False

    for c in claims[prop]:
        v = c["mainsnak"]["datavalue"]["value"]["id"]
        if v == qid:
            return True

    return False    
def sparql(query):

    r = session.get(
        SPARQL,
        params={"query": query, "format": "json"},
        timeout=60
    )

    r.raise_for_status()

    return r.json()
def safe_get(url, params, retries=5):

    for attempt in range(retries):

        try:
            r = session.get(url, params=params, timeout=30)

            if r.status_code == 200:
                return r

        except requests.exceptions.RequestException as e:
            print("HTTP error:", e)

        time.sleep(2 * (attempt + 1))

    raise RuntimeError("API failed after retries")

def load_spa_cache():

    print("Loading SPA → Wikidata cache")

    query = """
    SELECT ?person ?spa WHERE {
      ?person wdt:P2488 ?spa .
    }
    """

    data = sparql(query)

    cache = {}
    for row in data["results"]["bindings"]:

        spa = row["spa"]["value"]
        qid = row["person"]["value"].split("/")[-1]

        cache[spa] = qid

    print("Loaded", len(cache), "SPA ids")

    return cache


def api_get(params):

    r = safe_get(COMMONS_API, params)
    r.raise_for_status()
 
    return r.json()
def get_category_batch(cmcontinue=None):

    params = {
        "action": "query",
        "format": "json",
        "generator": "categorymembers",
        "gcmtitle": CATEGORY,
        "gcmtype": "file",
        "gcmlimit": 50,

        "prop": "revisions|categories|pageprops",
        "rvprop": "content",
        "cllimit": "max"
    }

    if cmcontinue:
        params["gcmcontinue"] = cmcontinue

    data = api_get(params)

    pages = data.get("query", {}).get("pages", {})
    cont = None
    if "continue" in data:
        cont = data["continue"]["gcmcontinue"]

    return pages, cont


category_qid_cache = {}

def get_category_qid(cat):

    if not cat.startswith("Category:"):
        return None
    if cat in category_qid_cache:
        return category_qid_cache[cat]
    
    params = {
        "action": "query",
        "format": "json",
        "prop": "pageprops",
        "titles": cat
    }

    data = api_get(params)

    qid = None
    for p in data["query"]["pages"].values():
        qid = p.get("pageprops", {}).get("wikibase_item")

    category_qid_cache[cat] = qid
    return qid


import time
human_cache = {}
def is_human(qid):

    if qid in human_cache:
        return human_cache[qid]

    params = {
        "action": "wbgetentities",
        "format": "json",
        "ids": qid,
        "props": "claims"
    }

    for attempt in range(3):

        r = session.get(WD_API, params=params, timeout=30)
        time.sleep(0.02)
        if not r.text.strip():
            time.sleep(2)
            continue

        if r.status_code != 200:
            time.sleep(2)
            continue

        try:
            data = r.json()
        except ValueError:
            print("Bad JSON from Wikidata:", r.text[:200])
            time.sleep(2)
            continue

        entity = data.get("entities", {}).get(qid)
        
        if not entity:
            human_cache[qid] = False
            return False
        
        claims = entity.get("claims", {})

        if "P31" not in claims:
            human_cache[qid] = False
            return False

        for c in claims["P31"]:
            if c["mainsnak"]["datavalue"]["value"]["id"] == "Q5":
                human_cache[qid] = True
                return True

        human_cache[qid] = False
        return False

    human_cache[qid] = False
    return False

print("Loading SPA cache…")

spa_cache = load_spa_cache()
print("Loading Commons category…")

cmcontinue = None

while True:

    try:
        pages, cmcontinue = get_category_batch(cmcontinue)
    except Exception as e:
        print("Batch failed:", e)
        time.sleep(5)
        continue

    mids = [f"M{p['pageid']}" for p in pages.values()]
    get_mediainfo_batch(mids)

    for p in pages.values():

        text = p.get("revisions", [{}])[0].get("*", "")
        m = SPA_REGEX.search(text)

        if not m:
            continue

        spa = m.group(1)

        pageid = p["pageid"]
        mid = f"M{pageid}"

        claims = mediainfo_cache.get(mid, {})

        if not has_string_claim(claims, "P4819", spa):
            qs.append(f'{mid}|P4819|"{spa}"')
        else:
            skipped_spa += 1

        if spa in spa_cache:

            qid = spa_cache[spa]

            if not has_item_claim(claims, "P180", qid):
                qs.append(f"{mid}|P180|{qid}")
            else:
                skipped_depicts += 1

            continue

        cats = p.get("categories", [])

        for c in cats:

            cat = c["title"]
            qid = get_category_qid(cat)

            if not qid:
                continue

            if is_human(qid):

                if not has_item_claim(claims, "P180", qid):
                    qs.append(f"{mid}|P180|{qid}")
                else:
                    skipped_depicts += 1

                break

    with open(QS_FILE, "w") as f:
        f.write("\n".join(qs))
    
    save_state(len(qs), qs, skipped_spa, skipped_depicts)
    
    print(
        "QS:", len(qs),
        "Skipped SPA:", skipped_spa,
        "Skipped depicts:", skipped_depicts
    )

    if not cmcontinue:
        break

    time.sleep(0.05)
print("Statements:", len(qs)) 
print("Unique categories checked:", len(category_qid_cache))
print("Unique QIDs checked:", len(human_cache))
print("Skipped SPA:", skipped_spa)
print("Skipped depicts:", skipped_depicts)

Loading SPA cache…
Loading SPA → Wikidata cache
Loaded 4480 SPA ids
Loading Commons category…
QS: 100 Skipped SPA: 0 Skipped depicts: 0
QS: 198 Skipped SPA: 0 Skipped depicts: 0
QS: 296 Skipped SPA: 0 Skipped depicts: 0
QS: 394 Skipped SPA: 0 Skipped depicts: 0
QS: 492 Skipped SPA: 0 Skipped depicts: 0
QS: 589 Skipped SPA: 0 Skipped depicts: 0
QS: 687 Skipped SPA: 0 Skipped depicts: 0
QS: 787 Skipped SPA: 0 Skipped depicts: 0
QS: 880 Skipped SPA: 0 Skipped depicts: 0
QS: 979 Skipped SPA: 0 Skipped depicts: 0
QS: 1079 Skipped SPA: 0 Skipped depicts: 0
QS: 1171 Skipped SPA: 0 Skipped depicts: 0
QS: 1267 Skipped SPA: 0 Skipped depicts: 0
QS: 1364 Skipped SPA: 0 Skipped depicts: 0
QS: 1464 Skipped SPA: 0 Skipped depicts: 0
QS: 1559 Skipped SPA: 0 Skipped depicts: 0
QS: 1656 Skipped SPA: 0 Skipped depicts: 0
QS: 1752 Skipped SPA: 0 Skipped depicts: 0
QS: 1852 Skipped SPA: 0 Skipped depicts: 0
QS: 1948 Skipped SPA: 0 Skipped depicts: 0
QS: 2043 Skipped SPA: 0 Skipped depicts: 0
QS: 2141 Skip

In [3]:
clean = []

for line in qs:
    if "|P180|" not in line:
        continue
    clean.append(f"{line}")

qsfileclean = "quickstatements_depicts_clean_" + version + ".txt"
with open("quickstatements_depicts_clean.txt", "w") as f:
    f.write("\n".join(clean))

In [4]:
end = datetime.now()
print("Ended: ", end) 
print('Time elapsed (hh:mm:ss.ms) {}'.format(datetime.now() - start_time))

Ended:  2026-03-07 22:20:07.815504
Time elapsed (hh:mm:ss.ms) 1:01:43.640492
